# Scripting with simcore

The web app covers the everyday work: one flight, a Monte Carlo campaign, the go/no-go
board, ballast. Everything it does is a call into `simcore`, and you can make the same
calls yourself for things the app doesn't do: sweeps, trade studies, your own plots,
batch runs over many missions, fitting to flight data.

This notebook goes from loading a mission to a parallel Monte Carlo campaign, using
Calisto, the reference rocket in `missions/examples/`. Each section stands on its own
once the setup cells have run.

Run it from the repo root:

```sh
uv sync --group notebooks
uv run --group notebooks jupyter lab notebooks/
```

**Units:** SI everywhere, except angles in degrees (ADR 0001). Positions along the rocket
are metres aft of the nose tip (ADR 0002). Field names carry their units (`_m`, `_kg`,
`_ms`, `_deg`), and every field has a description: `Mission.model_fields["site"]`, or
`docs/reference/mission-config.md`.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The repo root, wherever Jupyter was started from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "missions").is_dir())
OUT = ROOT / "data" / "notebook"  # data/ is gitignored
OUT.mkdir(parents=True, exist_ok=True)

## 1. Load a mission

A mission is a `Mission` (a Pydantic model) plus an **asset store** holding the files it
references by sha256: thrust curves and drag tables. There are two ways in.

**From a `.mission.zip`**, as exported from the app: the YAML and every asset, verified
against their hashes.

In [ ]:
from simcore.assets import MemoryAssetStore
from simcore.bundle import read_bundle
from simcore.mission import validate_mission

raw, files = read_bundle((ROOT / "missions/examples/calisto.mission.zip").read_bytes())
assets = MemoryAssetStore(files)

mission, report = validate_mission(raw)
print("valid:", report.valid)
for issue in report.issues:
    print(f"  {issue.severity}: {issue.path}: {issue.message}")

**From a YAML file** with its assets beside it under their own names. `assets_from_directory`
hashes every file in the directory, so the names don't matter. This is how
`python -m simcore run` works.

In [ ]:
from simcore.assets import assets_from_directory
from simcore.mission import parse_yaml

path = ROOT / "tests/fixtures/calisto/calisto.mission.yaml"
mission2, report2 = validate_mission(parse_yaml(path.read_text()))
assets2 = assets_from_directory(path.parent)
print(mission2.meta.name, "| valid:", report2.valid)

`validate_mission` returns the mission (or `None` if it can't be parsed) and a report of
errors and warnings, the same ones the app shows. Use `load_mission(raw)` instead when you
want an exception rather than a report.

Missions are ordinary Pydantic models, so you can read any field:

In [ ]:
r = mission.rocket
print(f"dry mass {r.dry_mass_kg} kg, CG {r.dry_cg_m} m aft of the tip, radius {r.radius_m} m")
print(f"motor: {mission.motor.kind}, burn {mission.motor.burn_time_s} s")
site = mission.site
print(f"site: {site.latitude_deg}, {site.longitude_deg}, elevation {site.elevation_m} m")
print(f"rail: {site.rail_length_m} m at {site.inclination_deg}°, heading {site.heading_deg}°")
for d in mission.recovery.devices:
    print(f"  {d.name}: CdS {d.cd_s_m2} m², trigger {d.trigger}")

## 2. Fly it

`run_flight` runs one deterministic flight and returns a `FlightResult`: a `summary` of
scalars and a `trajectory` of NumPy arrays. No RocketPy objects come back.

In [ ]:
from simcore.engine import TRAJECTORY_CHANNELS, run_flight

result = run_flight(mission, assets)
s = result.summary
print(f"apogee        {s.apogee_agl_m:8.1f} m AGL at {s.apogee_time_s:.1f} s")
print(f"rail exit     {s.rail_exit_velocity_ms:8.1f} m/s")
print(f"margin        {s.stability_margin_at_rail_exit_cal:8.2f} cal at rail exit")
print(f"max Mach      {s.max_mach:8.2f}")
print(
    f"landing       {s.landing.drift_distance_m:8.1f} m from the pad, "
    f"at {s.landing.speed_ms:.1f} m/s"
)
for w in result.warnings:
    print("warning:", w)

The whole summary is a model, so `model_dump()` gives a dict (handy for pandas or JSON).
Every field's units and meaning are in its description:

In [ ]:
from simcore.engine import FlightSummary

for name, value in s.model_dump(exclude={"landing", "parachute_events"}).items():
    units = FlightSummary.model_fields[name].json_schema_extra["units"]
    print(f"{name:36} {value:10.4g} {units}")

The trajectory channels, and a plot:

In [ ]:
for name, (units, description) in TRAJECTORY_CHANNELS.items():
    print(f"{name:24} {units:5} {description}")

In [ ]:
t = result.trajectory["time_s"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(t, result.trajectory["altitude_agl_m"])
for e in s.parachute_events:
    ax1.axvline(e.triggered_s, ls="--", c="grey")
    ax1.annotate(
        e.name, (e.triggered_s, e.triggered_agl_m), xytext=(5, 5), textcoords="offset points"
    )
ax1.set(xlabel="time (s)", ylabel="altitude AGL (m)", title="Altitude")
ax2.plot(result.trajectory["east_m"], result.trajectory["north_m"])
ax2.plot(0, 0, "k^", label="pad")
ax2.set(xlabel="east (m)", ylabel="north (m)", title="Ground track", aspect="equal")
ax2.legend();

Pass `terminate_on_apogee=True` when you only need the ascent: it is about 30% faster, and
the result has no landing. The sweeps below use it.

## 3. Change the mission

Never edit a mission in place: copy it with `model_copy(deep=True)`, then change the copy.
Every field is validated on construction, not on assignment, so re-validate
when you build a mission from your own data (see section 9).

`mission_hash` is the config's identity: the app keys stored results on it.

In [ ]:
from simcore.mission import mission_hash

heavier = mission.model_copy(deep=True)
heavier.rocket.dry_mass_kg += 0.5
print(mission_hash(mission)[:12], "->", mission_hash(heavier)[:12])
ascent = run_flight(heavier, assets, terminate_on_apogee=True)
print("apogee:", round(ascent.summary.apogee_agl_m), "m")

`simcore.mission.diff` compares two missions field by field, as the app's compare view does:

In [ ]:
from simcore.mission.diff import diff_missions

for change in diff_missions(mission.model_dump(mode="json"), heavier.model_dump(mode="json")):
    print(f"{change.path}: {change.a} -> {change.b}")

## 4. Sweeps

A sweep is a loop. Here: stability margin against fin span, a trade study the app
doesn't do. Each point is one ascent-only flight.

In [ ]:
spans = np.linspace(0.09, 0.15, 7)
at_rail, lowest = [], []
for span in spans:
    m = mission.model_copy(deep=True)
    m.rocket.fins[0].span_m = float(span)
    s_ = run_flight(m, assets, terminate_on_apogee=True).summary
    at_rail.append(s_.stability_margin_at_rail_exit_cal)
    lowest.append(s_.min_stability_margin_cal)

plt.plot(spans * 1000, at_rail, "o-", label="at rail exit")
plt.plot(spans * 1000, lowest, "s-", label="lowest in flight")
plt.axhline(1.5, ls="--", c="grey", lw=0.8)
plt.xlabel("fin span (mm)")
plt.ylabel("stability margin (cal)")
plt.legend();

Two things to know when sweeping geometry:

- **Drag comes from the mission's drag tables**, not from the geometry, so apogee here barely
  moves with fin span. If drag matters to the study, scale `rocket.drag.multiplier` with it
  (or supply a table per variant).
- **An unstable rocket can take minutes to simulate** as it tumbles (below about 85 mm span
  here). Keep sweeps in the stable range, or screen each variant first with
  `simcore.engine.center_of_pressure_m(mission, assets, mach)`, which needs no flight.

### Wind

The mission's atmosphere is `environment.atmosphere`: the standard atmosphere, a manual wind
profile, or a resolved forecast profile. Here is drift against a constant wind from the
west, full flights this time because drift needs the descent.

`direction_deg` is where the wind blows **from**, clockwise from true north.

In [ ]:
from simcore.mission.models import ManualProfile, ManualWindLevel


def with_wind(m, speed_ms, from_deg=270.0):
    m = m.model_copy(deep=True)
    m.environment.atmosphere = ManualProfile(
        kind="manual",
        note="notebook sweep",
        levels=[ManualWindLevel(height_m=0, speed_ms=speed_ms, direction_deg=from_deg)],
    )
    return m


winds = [0, 2, 4, 6, 8, 10]
flights = [run_flight(with_wind(mission, v), assets) for v in winds]

fig, ax = plt.subplots(figsize=(7, 5))
for v, f in zip(winds, flights, strict=True):
    ax.plot(f.trajectory["east_m"], f.trajectory["north_m"], label=f"{v} m/s")
ax.plot(0, 0, "k^")
ax.set(
    xlabel="east (m)", ylabel="north (m)", title="Ground tracks, wind from the west", aspect="equal"
)
ax.legend(title="wind");

Rail inclination and heading are on `mission.site`, so a weathercocking study is the same
loop over `inclination_deg` and `heading_deg`.

## 5. Monte Carlo

A campaign varies the inputs in `mission.monte_carlo.parameters` (each a `Spread` with a
`sigma` and an `enabled` flag) and flies every sample. `run_samples` flies a list of sample
indices and returns plain rows; `summarize` turns rows into statistics and landing
ellipses. The same seed and indices give the same samples, so a campaign can be split
across processes and the rows concatenated.

In [ ]:
from simcore.montecarlo import run_samples, summarize

mc = mission.model_copy(deep=True)
mc.monte_carlo.samples = 40  # keep it quick; the app defaults to 200
mc.monte_carlo.seed = 7
mc.monte_carlo.parameters.wind_layers.enabled = False  # no wind in this mission anyway
mc = with_wind(mc, 5.0)

for name, spread in mc.monte_carlo.parameters:
    print(f"{name:18} sigma {spread.sigma:<6} {'on' if spread.enabled else 'off'}")

Fly the samples in parallel, one chunk per process. Everything passed to the workers
(missions, the asset store, `run_samples` itself) pickles, so a standard
`ProcessPoolExecutor` is all it takes.

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor
from itertools import repeat

from simcore.montecarlo.store import chunks

n = mc.monte_carlo.samples
workers = min(os.cpu_count() or 1, 8)
batches = chunks(n, size=max(1, n // workers))
with ProcessPoolExecutor(workers) as pool:
    rows = [
        row for batch in pool.map(run_samples, repeat(mc), repeat(assets), batches) for row in batch
    ]

summary = summarize(mc, rows)
print(f"{summary.done} flown, {summary.failed} failed")
for stat in summary.stats:
    if stat.name in {"apogee_agl_m", "rail_exit_velocity_ms", "drift_m", "landing_speed_ms"}:
        print(
            f"{stat.label:28} p5 {stat.p5:8.1f}  p50 {stat.p50:8.1f}  "
            f"p95 {stat.p95:8.1f} {stat.units}"
        )

Each row has the sample's inputs as well as its outputs, so you can ask questions the
summary doesn't answer, like which input drives apogee:

In [ ]:
ok = [r for r in rows if r["error"] is None]
inputs = ["impulse_scale", "burn_time_scale", "dry_mass_kg", "drag_multiplier", "inclination_deg"]
apogees = np.array([r["apogee_agl_m"] for r in ok])
for name in inputs:
    x = np.array([r[name] for r in ok])
    print(f"{name:18} correlation with apogee {np.corrcoef(x, apogees)[0, 1]:+.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.hist(apogees, bins=15)
ax1.set(xlabel="apogee AGL (m)", ylabel="samples", title="Apogee")

east = [r["landing_east_m"] for r in ok]
north = [r["landing_north_m"] for r in ok]
ax2.scatter(east, north, s=10, label="landings")
lat0, lon0 = mc.site.latitude_deg, mc.site.longitude_deg
k_lat = 111_320.0
k_lon = k_lat * np.cos(np.radians(lat0))
for ellipse in summary.ellipses:
    lon, lat = np.array(ellipse.polygon + ellipse.polygon[:1]).T
    ax2.plot((lon - lon0) * k_lon, (lat - lat0) * k_lat, label=f"{ellipse.probability:.1%}")
ax2.plot(0, 0, "k^")
ax2.set(xlabel="east (m)", ylabel="north (m)", title="Landing dispersion", aspect="equal")
ax2.legend();

## 6. Go/no-go board

The board judges the mission's rules (a competition profile, overrides, and the team's own)
on the nominal flight and the campaign. Choose a profile on a copy of the mission:

In [ ]:
from simcore.constraints import board, from_row, from_summary, load_profiles

for pid, profile in load_profiles().items():
    print(pid, "|", profile.name, "|", [c.id for c in profile.categories])

In [ ]:
judged = mc.model_copy(deep=True)
judged.constraints.profile = "aurc-2026"
judged.constraints.category = "10k-srad-hybrid"

nominal = from_summary(run_flight(judged, assets).summary)
b = board(judged.constraints, nominal, [from_row(r) for r in ok])
print("verdict:", b.verdict)
for row in b.rows:
    value = row.value_low if row.value_low is not None else row.value_high
    print(
        f"  {row.outcome:8} {row.label:52} "
        f"{value if value is None else round(value, 2)!s:>10} {row.units}"
    )
if b.score:
    print(f"expected score {b.score.expected:.1f} of {b.score.max_points}")

Calisto's drogue is too small for this profile's 20 m/s minimum, so the board says no-go.
Team rules go in `constraints.additional` as `Constraint`s; `simcore.constraints.METRICS`
lists what a rule can test.

## 7. Ballast

`simcore.ballast.solve` finds the ballast that puts apogee on target, rounds it to your
increment, and checks the answer with a full flight and a small campaign. It needs the bay
position and a target (or a competition category with one). Slow: about `samples + 10`
flights.

In [ ]:
from simcore.ballast.solve import solve

trim = mission.model_copy(deep=True)
trim.ballast.position_m = 0.6
trim.ballast.target_apogee_m = 3000.0
trim.ballast.increment_kg = 0.1
trim.ballast.max_kg = 3.0

sol = solve(trim, assets, samples=20)
print(sol.status, "| exact", round(sol.exact_kg, 3), "kg | recommended", sol.recommended_kg, "kg")
print(
    f"apogee {sol.apogee_agl_m:.0f} m (p5 {sol.apogee_p5_m:.0f}, p95 {sol.apogee_p95_m:.0f}), "
    f"{sol.gradient_m_per_100g:.1f} m per 100 g"
)

pts = sorted(sol.curve, key=lambda p: p.mass_kg)
plt.plot([p.mass_kg for p in pts], [p.apogee_agl_m for p in pts], "o-")
plt.axhline(sol.target_apogee_m, ls="--", c="grey")
plt.xlabel("ballast (kg)")
plt.ylabel("apogee AGL (m)");

To fly with ballast, set `ballast.installed_kg` (every flight includes it) or use
`simcore.ballast.with_ballast(mission, kg)` for a copy with the mass folded into the rocket.

## 8. Recovery and post-flight

Steady descent rate under each device, and the CdS that would give a target rate:

In [ ]:
from simcore.recovery import descent_table

for d in descent_table(mission, s.apogee_agl_m, target_rates_ms={"Main": 6.0}):
    target = (
        f", {d.cd_s_for_target_m2:.1f} m² for {d.target_rate_ms} m/s" if d.target_rate_ms else ""
    )
    print(
        f"{d.name:7} CdS {d.cd_s_m2:5.1f} m²: "
        f"{d.rate_bottom_ms:5.1f} m/s at {d.bottom_agl_m:.0f} m{target}"
    )

**Fitting to a flight log.** `read_log` reads an altimeter CSV; `fit` adjusts the drag
multiplier (and optionally the impulse) until the simulated ascent matches it. Here the
"log" is synthetic: a flight with 15% more drag, sampled at 20 Hz with noise and a pad
wait, so we know the answer the fit should find.

In [ ]:
from simcore.postflight import read_log
from simcore.postflight.fit import fit

truth = mission.model_copy(deep=True)
truth.rocket.drag.multiplier = 1.15
flown = run_flight(truth, assets, terminate_on_apogee=True).trajectory

rng = np.random.default_rng(0)
log_t = np.arange(0, flown["time_s"][-1] + 3, 0.05)
log_alt = np.interp(log_t - 3, flown["time_s"], flown["altitude_agl_m"], left=0) + rng.normal(
    0, 2, len(log_t)
)
csv = "time,alt_ft\n" + "\n".join(
    f"{t:.2f},{a / 0.3048:.1f}" for t, a in zip(log_t, log_alt, strict=True)
)

t_log, alt_log = read_log(csv.encode(), "time", "alt_ft", "ft")
fitted = fit(mission, assets, t_log, alt_log)
print(
    f"drag multiplier {fitted.drag_multiplier_before:.2f} -> {fitted.drag_multiplier:.3f}, "
    "truth 1.15"
)
print(
    f"RMS error {fitted.rms_before_m:.1f} m -> {fitted.rms_after_m:.1f} m, {fitted.flights} flights"
)

for curve, label in [
    (fitted.measured, "log"),
    (fitted.predicted, "before"),
    (fitted.fitted, "fitted"),
]:
    plt.plot(curve.t, curve.altitude_agl_m, label=label)
plt.xlabel("time since launch (s)")
plt.ylabel("altitude AGL (m)")
plt.legend();

## 9. Build a mission in code, and save it

A mission can come from anywhere: a spreadsheet, an OpenRocket file
(`simcore.importers.openrocket.import_openrocket`), or a dict. Assets go into a store by
content; the mission refers to them by the hash `put` returns. `scripts/lanius_test_mission.py`
is a complete example built from a `.ork`.

Here: Calisto with an AeroTech M1340W in place of its own motor, as a generic motor whose
masses are estimated from the thrust curve.

In [ ]:
from simcore.importers.thrust import generic_motor_estimate, parse_eng

eng = (ROOT / "tests/fixtures/motors/AeroTech_M1340W.eng").read_bytes()
sha = assets.put(eng)

data = mission.model_dump(mode="json")
data["meta"]["name"] = "Calisto, M1340W"
estimate = generic_motor_estimate(parse_eng(eng.decode()))
data["motor"] = {"kind": "generic", "thrust_curve": sha, **estimate}
data["assets"][sha] = {"kind": "thrust_curve", "filename": "AeroTech_M1340W.eng", "size": len(eng)}

swapped, report = validate_mission(data)
for issue in report.issues:
    print(f"{issue.severity}: {issue.path}: {issue.message}")
print("valid:", report.valid)
ascent = run_flight(swapped, assets, terminate_on_apogee=True)
print("apogee:", round(ascent.summary.apogee_agl_m), "m")

Save it as YAML (with its assets beside it, so `python -m simcore run` works) or as a
`.mission.zip` to import in the app under **Missions**:

In [ ]:
from simcore.bundle import write_bundle
from simcore.mission import dump_yaml

(OUT / "calisto-m1340w.mission.yaml").write_text(dump_yaml(swapped))
(OUT / "AeroTech_M1340W.eng").write_bytes(eng)
(OUT / "calisto-m1340w.mission.zip").write_bytes(write_bundle(swapped, assets))
print(sorted(p.name for p in OUT.iterdir()))

## Where next

- `docs/methods/` says what each computation assumes and where it is weak. Read the page
  for anything you build on.
- `docs/reference/mission-config.md` lists every mission field with its units.
- `python -m simcore validate|hash|run|bundle mission.yaml` does the common cases from a
  shell.
- Only `simcore.engine` touches RocketPy, and nothing it returns is a RocketPy object. If you
  need RocketPy itself, `simcore.engine.build.build_rocket` and friends build the objects
  from a mission, but results from there aren't what the app would report.